# 9.1 퍼셉트론의 한계, 순전파, 역전파 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter09_1_backprop_xor.ipynb)

책 본문: [9.1 퍼셉트론의 한계, 순전파, 역전파](https://smhanlab.com/book-ml/kor/ml1/chapter09/1.html)

이 노트북은 9.1절의 수치를 코드로 재현합니다:
(1) 본문 손계산 예제(2-2-1, x=(1,0), y=1)의 순전파/역전파를 그대로 계산,
(2) **수치 미분**과 **자동 미분(torch)**으로 역전파를 서로 교차 검증,
(3) 은닉 유닛 4개 신경망으로 XOR을 실제로 학습,
(4) 은닉층 없는 로지스틱회귀 / 0-초기화 / 학습률 비교 실험,
(5) 입력 공간과 은닉 공간의 결정경계를 시각화합니다.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
np.random.seed(42)

## 0. 퍼셉트론의 sgn(z) vs 로지스틱회귀의 시그모이드

본문 도입부 그림(`ch09_1_sgn_vs_sigmoid.svg`)을 재현합니다: 퍼셉트론은 시그모이드
대신 스텝함수 sgn(z)를 쓴다는 것 하나가 로지스틱회귀와의 유일한 차이입니다.

In [2]:
def sgn(z):
    return np.where(z >= 0, 1.0, -1.0)

def sigmoid_(z):
    return 1.0 / (1.0 + np.exp(-z))

zz = np.linspace(-5, 5, 1001)
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(zz, sigmoid_(zz), lw=2, color="#d95f02", ls="--", label=r"$\sigma(z)$ (logistic regression)")
ax.plot(zz, sgn(zz), lw=2.2, color="#4878a8", label=r"$\mathrm{sgn}(z)$ (perceptron)")
ax.scatter([0], [1], s=28, color="#4878a8", zorder=5)
ax.scatter([0], [-1], s=28, facecolors="white", edgecolors="#4878a8", linewidths=1.5, zorder=5)
ax.axhline(0, color="0.8", lw=0.5); ax.axvline(0, color="0.8", lw=0.5)
ax.set_xlabel(r"$z = w^Tx + b$"); ax.set_ylabel(r"$\hat{y}$")
ax.set_yticks([-1, 0, 1])
ax.set_title(r"$\mathrm{sgn}(z)$ vs. $\sigma(z)$ — 연속 확률이냐, 이산 임계값이냐")
ax.legend(fontsize=9, loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch09_1_sgn_vs_sigmoid.svg", bbox_inches="tight")
plt.show()
print("figure saved -> kor/src/images/ch09_1_sgn_vs_sigmoid.svg")

figure saved -> kor/src/images/ch09_1_sgn_vs_sigmoid.svg


## 1. 순전파: 본문 손계산 예제 (2-2-1, x=(1,0), y=1)

본문의 가중치를 그대로 대입합니다. 컨벤션: `W1`의 **행=입력 차원(2), 열=은닉 유닛(2)**,
`z1 = x @ W1 + b1`. 본문에서 `W1 = [[0.5, 0.3], [-0.5, 0.8]]`, `b1 = (0.1, -0.2)`,
`W2 = (0.7, -0.6)`, `b2 = 0.1`입니다.

In [3]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

x = np.array([1.0, 0.0]); y = 1.0
W1 = np.array([[0.5, 0.3], [-0.5, 0.8]])   # 행=입력, 열=은닉유닛 (본문 손계산 예제)
b1 = np.array([0.1, -0.2])
W2 = np.array([0.7, -0.6])
b2 = 0.1

z1 = x @ W1 + b1; a1 = sigmoid(z1)
z2 = a1 @ W2 + b2; a2 = sigmoid(z2)
L = 0.5 * (a2 - y) ** 2
print(f"z1 = {z1}   a1 = {np.round(a1, 4)}")
print(f"z2 = {z2:.4f}   a2 = {a2:.4f}")
print(f"L  = {L:.5f}")
assert np.allclose(z1, [0.6, 0.1])
assert abs(a1[0] - 0.646) < 1e-3 and abs(a1[1] - 0.525) < 1e-3
assert abs(z2 - 0.237) < 1e-3 and abs(a2 - 0.559) < 1e-3 and abs(L - 0.097) < 1e-3
print("본문 수치(0.6, 0.1 / 0.646, 0.525 / 0.237 / 0.559 / 0.097)와 일치 ✓")

z1 = [0.6 0.1]   a1 = [0.6457 0.525 ]
z2 = 0.2370   a2 = 0.5590
L  = 0.09725
본문 수치(0.6, 0.1 / 0.646, 0.525 / 0.237 / 0.559 / 0.097)와 일치 ✓


## 2. 역전파: δ와 그래디언트 계산

본문의 세 줄: \(\delta_2 = (a_2-y)\sigma'(z_2)\),
\(\delta_1 = (W_2\,\delta_2) \odot \sigma'(z_1)\),
그래디언트 \(\partial L/\partial W_2 = a_1\,\delta_2\),
\(\partial L/\partial W_1 = x^{\top}\delta_1\) (외적).

In [4]:
d2 = (a2 - y) * a2 * (1 - a2)                      # δ2
d1 = (W2 * d2) * (a1 * (1 - a1))                   # δ1 (W2δ2 ∘ σ′(z1))
gW2 = d2 * a1                                      # ∂L/∂W2
gb2 = d2                                           # ∂L/∂b2
gW1 = np.outer(x, d1)                              # ∂L/∂W1 = x ⊗ δ1
gb1 = d1                                           # ∂L/∂b1

print(f"δ2 = {d2:.5f}   δ1 = {np.round(d1, 4)}")
print(f"∂L/∂W2 = {np.round(gW2, 4)}   ∂L/∂b2 = {gb2:.5f}")
print(f"∂L/∂W1 =\n{np.round(gW1, 4)}")
print(f"∂L/∂b1 = {np.round(gb1, 4)}")
assert abs(d2 - (-0.109)) < 1e-3
assert np.allclose(np.round(d1, 3), [-0.017, 0.016])
assert np.allclose(np.round(gW2, 3), [-0.070, -0.057])

δ2 = -0.10872   δ1 = [-0.0174  0.0163]
∂L/∂W2 = [-0.0702 -0.0571]   ∂L/∂b2 = -0.10872
∂L/∂W1 =
[[-0.0174  0.0163]
 [-0.      0.    ]]
∂L/∂b1 = [-0.0174  0.0163]


### 학습률 α=0.5로 한 스텝 갱신 → 손실 감소

본문의 마지막 문장("0.097 → 0.087")을 확인합니다.

In [5]:
alpha = 0.5
W1n = W1 - alpha * gW1; b1n = b1 - alpha * gb1
W2n = W2 - alpha * gW2; b2n = b2 - alpha * gb2
a1n = sigmoid(x @ W1n + b1n); a2n = sigmoid(a1n @ W2n + b2n)
Lnew = 0.5 * (a2n - y) ** 2
print(f"L: {L:.5f} -> {Lnew:.5f}   (a2: {a2:.4f} -> {a2n:.4f})")
assert Lnew < L
print("한 스텝의 역전파+경사하강으로 은닉층 가중치 W1까지 갱신, 손실 감소 ✓")

L: 0.09725 -> 0.08703   (a2: 0.5590 -> 0.5828)
한 스텝의 역전파+경사하강으로 은닉층 가중치 W1까지 갱신, 손실 감소 ✓


## 3. 수치 미분으로 역전파 검증

각 파라미터를 \(\epsilon = 10^{-6}\)씩 흔들어서
\(\frac{L(w+\epsilon)-L(w-\epsilon)}{2\epsilon}\)를 계산한 뒤,
역전파의 해석적 그래디언트와 비교합니다. **역전파가 맞다는 것을
기계로 확인하는 표준 기법**입니다. (주의: 수치 미분 코드도 본문과
같은 컨벤션 — `W1` 행=입력 — 을 써야 전치로 인한 오차가 생기지
않습니다.)

In [6]:
eps = 1e-6
def loss_of(W1, b1, W2, b2):
    a1_ = sigmoid(x @ W1 + b1)
    return 0.5 * (sigmoid(a1_ @ W2 + b2) - y) ** 2

# 파라미터별 수치 그래디언트
def num_grads():
    nW1 = np.zeros_like(W1)
    for i in range(2):
        for j in range(2):
            Wp = W1.copy(); Wm = W1.copy(); Wp[i, j] += eps; Wm[i, j] -= eps
            nW1[i, j] = (loss_of(Wp, b1, W2, b2) - loss_of(Wm, b1, W2, b2)) / (2 * eps)
    nb1 = np.zeros_like(b1)
    for i in range(2):
        bp = b1.copy(); bm = b1.copy(); bp[i] += eps; bm[i] -= eps
        nb1[i] = (loss_of(W1, bp, W2, b2) - loss_of(W1, bm, W2, b2)) / (2 * eps)
    nW2 = np.zeros_like(W2)
    for i in range(2):
        Wp = W2.copy(); Wm = W2.copy(); Wp[i] += eps; Wm[i] -= eps
        nW2[i] = (loss_of(W1, b1, Wp, b2) - loss_of(W1, b1, Wm, b2)) / (2 * eps)
    nb2 = (loss_of(W1, b1, W2, b2 + eps) - loss_of(W1, b1, W2, b2 - eps)) / (2 * eps)
    return nW1, nb1, nW2, nb2

nW1, nb1, nW2, nb2 = num_grads()
print(f"최대 절대오차:  W1={np.abs(nW1-gW1).max():.2e}  b1={np.abs(nb1-gb1).max():.2e}  "
      f"W2={np.abs(nW2-gW2).max():.2e}  b2={np.abs(nb2-gb2).max():.2e}")
assert max(np.abs(nW1-gW1).max(), np.abs(nb1-gb1).max(),
           np.abs(nW2-gW2).max(), np.abs(nb2-gb2).max()) < 1e-8
print("역전파 그래디언트 == 수치 미분 (10⁻¹¹ 수준 일치) ✓")

최대 절대오차:  W1=6.29e-12  b1=6.29e-12  W2=1.53e-11  b2=6.62e-12
역전파 그래디언트 == 수치 미분 (10⁻¹¹ 수준 일치) ✓


## 4. 자동 미분(torch)으로 교차 검증

같은 손실에 대해 `torch.autograd`의 `.backward()`로 그래디언트를 구하면,
역전파(연쇄법칙 재사용)가 자동으로 계산됩니다. 위에서 손으로(해석적으로)
구한 값과 비교합니다. 이것이 PyTorch/TensorFlow의 `.backward()`가
내부에서 하는 일입니다.

In [7]:
import torch
W1t = torch.tensor(W1, requires_grad=True); b1t = torch.tensor(b1, requires_grad=True)
W2t = torch.tensor(W2, requires_grad=True); b2t = torch.tensor(b2, requires_grad=True)
xt = torch.tensor(x); yt = torch.tensor(y)

z1t = xt @ W1t + b1t; a1t = torch.sigmoid(z1t)
z2t = a1t @ W2t + b2t; a2t = torch.sigmoid(z2t)
loss_t = 0.5 * (a2t - yt) ** 2
loss_t.backward()

print(f"torch 손실 = {loss_t.item():.5f}  (해석적 L = {L:.5f})")
print(f"torch ∂L/∂W1 =\n{np.round(W1t.grad.numpy(), 5)}")
print(f"해석적 ∂L/∂W1 =\n{np.round(gW1, 5)}")
print(f"torch ∂L/∂W2 = {np.round(W2t.grad.numpy(), 5)}   해석적 ∂L/∂W2 = {np.round(gW2, 5)}")
assert np.allclose(W1t.grad.numpy(), gW1, atol=1e-6)
assert np.allclose(W2t.grad.numpy(), gW2, atol=1e-6)
assert abs(b2t.grad.item() - gb2) < 1e-6
print("torch 자동 미분 == 우리 역전파 ✓  (.backward() 한 줄이 이 모든 것을 대신 계산)")

torch 손실 = 0.09725  (해석적 L = 0.09725)
torch ∂L/∂W1 =
[[-0.01741  0.01627]
 [-0.       0.     ]]
해석적 ∂L/∂W1 =
[[-0.01741  0.01627]
 [-0.       0.     ]]
torch ∂L/∂W2 = [-0.0702  -0.05708]   해석적 ∂L/∂W2 = [-0.0702  -0.05708]
torch 자동 미분 == 우리 역전파 ✓  (.backward() 한 줄이 이 모든 것을 대신 계산)


## 5. XOR을 실제로 풀기: 은닉 4개, 5000 에폭

XOR 데이터 4개(`[0,0]→0, [0,1]→1, [1,0]→1, [1,1]→0`)를 은닉 유닛 4개
신경망으로 학습합니다. 무작위 초기화(시드 고정), α=0.5, 배치=전체 4개.
손실 곡선을 10개 시점에 기록합니다.

In [8]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y = np.array([0.0, 1.0, 1.0, 0.0])

def train_xor(n_hidden, seed, epochs, alpha, init_scale=1.0):
    rng = np.random.RandomState(seed)
    W1_ = rng.randn(2, n_hidden) * init_scale     # 행=입력
    b1_ = rng.randn(n_hidden) * 0.1
    W2_ = rng.randn(n_hidden) * init_scale
    b2_ = 0.0
    rec = {}
    for ep in range(1, epochs + 1):
        z1 = X @ W1_ + b1_; a1 = sigmoid(z1)
        z2 = a1 @ W2_ + b2_; a2 = sigmoid(z2)
        loss_ = 0.5 * np.mean((a2 - Y) ** 2)
        if ep in (1, 10, 100, 500, 1000, 2500, 5000):
            rec[ep] = loss_
        d2_ = (a2 - Y) * a2 * (1 - a2)
        gW2_ = a1.T @ d2_; gb2_ = d2_.sum()
        d1_ = (d2_[:, None] * W2_[None, :]) * (a1 * (1 - a1))
        gW1_ = X.T @ d1_; gb1_ = d1_.sum(0)
        W1_ -= alpha * gW1_; b1_ -= alpha * gb1_
        W2_ -= alpha * gW2_; b2_ -= alpha * gb2_
    z1 = X @ W1_ + b1_; a1 = sigmoid(z1)
    a2 = sigmoid(a1 @ W2_ + b2_)
    return dict(W1=W1_, b1=b1_, W2=W2_, b2=b2_, preds=a2, rec=rec,
                final_loss=0.5 * np.mean((a2 - Y) ** 2))

res = train_xor(4, 7, 5000, 0.5)
print("손실 곡선 (에폭: 손실):")
for ep in sorted(res['rec']):
    print(f"  {ep:>5}: {res['rec'][ep]:.5f}")
print(f"최종 예측: {np.round(res['preds'], 4)}   반올림: {np.round(res['preds']).astype(int)}")
print(f"최종 손실: {res['final_loss']:.5f}")
assert list(np.round(res['preds']).astype(int)) == [0, 1, 1, 0]
print("XOR 패턴 (0,1,1,0) 정확히 재현 ✓")

손실 곡선 (에폭: 손실):
      1: 0.13566
     10: 0.12897
    100: 0.12418
    500: 0.06074
   1000: 0.00513
   2500: 0.00077
   5000: 0.00029
최종 예측: [0.0157 0.977  0.9755 0.0302]   반올림: [0 1 1 0]
최종 손실: 0.00029
XOR 패턴 (0,1,1,0) 정확히 재현 ✓


### 손실 곡선

In [9]:
eps_ = sorted(res['rec'])
plt.figure(figsize=(6.5, 3.5))
plt.semilogy(eps_, [res['rec'][e] for e in eps_], 'o-', color="#1d4ed8", lw=2)
plt.xlabel("Epoch"); plt.ylabel("Loss (log)")
plt.title("XOR learning curve — 4 hidden units, α=0.5, seed 7")
plt.grid(alpha=0.3); plt.tight_layout()
plt.show()

## 6. 결정적 증거: 로지스틱회귀(은닉층 없음)는 XOR을 못 푼다

같은 데이터, 같은 방식(경사하강법, α=1.0, 5000 에폭)으로 **은닉층이 없는**
로지스틱회귀를 학습시킵니다. 9.1절의 부등식 모순이 예측하듯
**모든 예측이 0.5(완전 불확신)에 멈춥니다**.

In [10]:
w = np.zeros(2); c = 0.0
lr = 1.0
for ep in range(5000):
    a = sigmoid(X @ w + c)
    g = X.T @ (a - Y) / len(Y)
    w -= lr * g; c -= lr * (a - Y).mean()
a = sigmoid(X @ w + c)
print(f"로지스틱회귀(은닉 없음) 최종 예측: {np.round(a, 4)}")
print(f"최종 손실: {0.5*np.mean((a-Y)**2):.5f}   (w={np.round(w,4)}, c={c:.4f})")
print(f"비교 — 2층 신경망(은닉 4개) 예측: {np.round(res['preds'], 4)}")
print("구조의 차이가 운명을 정한다: 은닉층 없이는 손실 0.125(=4×½·0.5²)에서 멈춤")

로지스틱회귀(은닉 없음) 최종 예측: [0.5 0.5 0.5 0.5]
최종 손실: 0.12500   (w=[0. 0.], c=0.0000)
비교 — 2층 신경망(은닉 4개) 예측: [0.0157 0.977  0.9755 0.0302]
구조의 차이가 운명을 정한다: 은닉층 없이는 손실 0.125(=4×½·0.5²)에서 멈춤


## 7. 초기화: 0-초기화는 대칭을 깨지 못해 멈춘다

가중치를 모두 0으로 초기화하면 모든 은닉 유닛이 동일한 출력을 내
**영원히 동일**한 업데이트를 받는다(대칭성 문제). 결과: 예측이 전부
0.5, 손실 0.125로 동결.

In [11]:
res0 = train_xor(4, 7, 5000, 0.5, init_scale=0.0)
print(f"0-초기화:  최종 예측 {np.round(res0['preds'], 4)}   손실 {res0['final_loss']:.5f}")
print(f"무작위초기: 최종 예측 {np.round(res['preds'], 4)}   손실 {res['final_loss']:.5f}")
assert np.allclose(res0['preds'], 0.5)
print("0-초기화 → 모든 은닉유닛 동일 출력 → 1차원 퇴화 → 멈춤 ✓")

0-초기화:  최종 예측 [0.5 0.5 0.5 0.5]   손실 0.12500
무작위초기: 최종 예측 [0.0157 0.977  0.9755 0.0302]   손실 0.00029
0-초기화 → 모든 은닉유닛 동일 출력 → 1차원 퇴화 → 멈춤 ✓


## 8. 학습률의 효과: "작으면 안전"이 아니다

같은 구조(은닉 4개, 시드 7, 5000 에폭)에서 학습률만 바꿉니다.
α=0.01은 사실상 학습이 진행되지 않는다(느리다 ≠ 못 배운다).

In [12]:
for a_ in (0.01, 0.5, 2.0):
    r = train_xor(4, 7, 5000, a_)
    print(f"α = {a_:<4}: 최종 손실 {r['final_loss']:.5f}   예측 {np.round(r['preds'], 4)}")
r01 = train_xor(4, 7, 5000, 0.01)
r05 = train_xor(4, 7, 5000, 0.5)
assert r01['final_loss'] > 0.1   # 사실상 무학습
assert r05['final_loss'] < 1e-3  # 성공
print("α=0.01: 5000에폭이 지나도 손실 ≈ 초기값 (Chapter 2.1의 '너무 작으면 느리다' 극단)")
print("α=0.5 : 수렴 성공. 적정 α는 문제마다 다르다.")

α = 0.01: 최종 손실 0.12411   예측 [0.4452 0.5103 0.5031 0.5549]


α = 0.5 : 최종 손실 0.00029   예측 [0.0157 0.977  0.9755 0.0302]


α = 2.0 : 최종 손실 0.00005   예측 [0.0066 0.9903 0.989  0.0134]


α=0.01: 5000에폭이 지나도 손실 ≈ 초기값 (Chapter 2.1의 '너무 작으면 느리다' 극단)
α=0.5 : 수렴 성공. 적정 α는 문제마다 다르다.


## 9. 결정경계 시각화: 입력 공간 vs 은닉 공간

**은닉 유닛 2개** 네트워크(20,000 에폭 학습 — 은닉 공간이 정확히 2차원이
되어야 (a₁,a₂) 평면에 진짜 결정경계를 그릴 수 있음)를 이용해
**입력 공간 (x₁,x₂)**과 **은닉 공간 (a₁,a₂)**의 결정경계를 그립니다.
왼쪽: 입력 공간에서는 비선형 곡선 경계. 오른쪽: 은닉 공간에서는
**직선**으로 분리된다 — "은닉층이 새로운 공간을 만들어 직선으로
나눈다"는 9.1절의 주장을 시각화합니다.
(SVG로 `kor/src/images/ch09_1_xor_boundary.svg`에 저장.)

In [13]:
vis = train_xor(2, 7, 20000, 0.5)
print(f"시각화용 2-hidden 모델: 예측 {np.round(vis['preds'],4)}, 손실 {vis['final_loss']:.5f}")
W1_, b1_, W2_, b2_ = vis['W1'], vis['b1'], vis['W2'], vis['b2']
def model_out(Xg, W1_, b1_, W2_, b2_):
    a1_ = sigmoid(Xg @ W1_ + b1_)
    return sigmoid(a1_ @ W2_ + b2_), a1_

g = np.linspace(-0.4, 1.4, 240)
gx, gy = np.meshgrid(g, g)
Xg = np.stack([gx.ravel(), gy.ravel()], axis=1)
a2g, _ = model_out(Xg, W1_, b1_, W2_, b2_)
a2g = a2g.reshape(gx.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
ax = axes[0]
ax.contourf(gx, gy, a2g, levels=20, cmap="RdBu_r", alpha=0.75)
ax.contour(gx, gy, a2g, levels=[0.5], colors="k", linestyles="--", lw=1.5)
for xi, yi, lab in [(0,0,0),(0,1,1),(1,0,1),(1,1,0)]:
    ax.scatter([xi],[yi], c=["#dc2626" if lab==0 else "#1d4ed8"],
               s=110, edgecolor="k", zorder=5)
ax.set_xlim(-0.4,1.4); ax.set_ylim(-0.4,1.4)
ax.set_xlabel("x₁"); ax.set_ylabel("x₂")
ax.set_title("Input space — not separable by a single line")
ax2 = axes[1]
# 은닉 공간: (a1,a2) 평면에서 4개 점 + 선형 결정경계
h1v, h2v = np.meshgrid(np.linspace(0,1,200), np.linspace(0,1,200))
z2h = W2_[0]*h1v + W2_[1]*h2v + b2_
ax2.contourf(h1v, h2v, z2h, levels=20, cmap="RdBu_r", alpha=0.75)
ax2.contour(h1v, h2v, z2h, levels=[0], colors="k", linestyles="--", lw=1.5)
a1pts = sigmoid(X @ W1_ + b1_)
for k,(xi,yi,lab) in enumerate([(0,0,0),(0,1,1),(1,0,1),(1,1,0)]):
    ax2.scatter([a1pts[k,0]],[a1pts[k,1]],
                c=["#dc2626" if lab==0 else "#1d4ed8"], s=110, edgecolor="k", zorder=5)
ax2.set_xlabel("a₁ (hidden unit 1)"); ax2.set_ylabel("a₂ (hidden unit 2)")
ax2.set_title("Hidden space — separable by a single line")
fig.tight_layout()
import os
os.makedirs(IMG, exist_ok=True)
fig.savefig(f"{IMG}/ch09_1_xor_boundary.svg", bbox_inches="tight")
plt.show()
print(f"그림 저장: {IMG}/ch09_1_xor_boundary.svg")

시각화용 2-hidden 모델: 예측 [0.013  0.989  0.989  0.0113], 손실 0.00007
그림 저장: /home/smhan/book-ml/kor/src/images/ch09_1_xor_boundary.svg


/tmp/ipykernel_1316331/1801929208.py:36: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_1316331/1801929208.py:36: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_1316331/1801929208.py:39: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Noto Sans CJK KR.
  fig.savefig(f"{IMG}/ch09_1_xor_boundary.svg", bbox_inches="tight")
/tmp/ipykernel_1316331/1801929208.py:39: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Noto Sans CJK KR.
  fig.savefig(f"{IMG}/ch09_1_xor_boundary.svg", bbox_inches="tight")


## 요약

- **순전파/역전파의 수치는 재현 가능**: 본문의 손계산 예제(2-2-1)를
  코드로 그대로 계산하면 \(z_1=(0.6,0.1)\), \(a_2\approx0.559\),
  \(L\approx0.097\), \(\delta_2\approx-0.109\)로 정확히 일치한다.
- **역전파는 정확하다**: 수치 미분(중심차분)과 torch 자동 미분 모두
  해석적 그래디언트와 \(10^{-11}\) 수준으로 일치한다. `.backward()` 한
  줄이 내부에서 하는 일이 바로 이 연쇄법칙 재사용이다.
- **XOR을 푸는 것**: 은닉 4개 + 무작위 초기화 + α=0.5, 5000 에폭이면
  예측이 (0.016, 0.977, 0.976, 0.030)로 정확히 XOR 패턴을 재현한다.
- **구조의 차이**: 은닉층 없는 로지스틱회귀는 같은 데이터에서
  (0.5,0.5,0.5,0.5)로 동결된다. 0-초기화도 대칭을 깨지 못해 동결된다.
- **시각화**: 입력 공간의 비선형 경계가 은닉 공간에서는 직선이 된다 —
  은닉층의 역할("새로운 공간을 만든다")을 직접 확인할 수 있다.

다음 9.2절에서는 이 구조를 깊이(층 수) 늘렸을 때 생기는
**그래디언트 소실·폭발** 문제를 다룬다 — 이번 절의 \(\delta_1\)에
곱해지는 \(\sigma'(z_1)\)이 층마다 반복되는 것이 그 시작점이다.